# ทดลอง conf: 0.15 / 0.30 / 0.50
เลือก Runtime → Change runtime type → T4 GPU แล้วรันจากบนลงล่าง

ใช้ **best.pt รอบแรก** กับ **Validation 36 ภาพ** เท่านั้น ไม่มีการเทรนและไม่อ่านภาพ/Label Test

อัปโหลด **football-v2-test-ready.zip ไฟล์เดิม** ที่มี best.pt และ Validation อยู่แล้ว ชื่อ ZIP มีคำว่า test แต่ Notebook นี้เลือกดึงเฉพาะ valid กับโมเดลออกมา

ภาพเปรียบเทียบจะมี 3 คอลัมน์: conf 0.15 / 0.30 / 0.50 เป็นโมเดลตัวเดียวกันทั้งหมด

ตารางนับ TP=ถูก, FP=เกิน, FN=ตกหล่น โดยกรอบต้องตรงคลาสและ IoU ≥ 0.50 จับคู่คำตอบได้ครั้งเดียวต่อวัตถุ เรียงผลทำนายจาก conf สูงไปต่ำ

Precision/Recall ใน Notebook นี้คำนวณที่ conf แต่ละค่าจริง ไม่ใช่ mAP และไม่ใช่ Precision/Recall ที่เลือกจุด max-F1 อัตโนมัติในรายงานเดิม

เพิ่ม conf อาจลดกรอบมั่ว แต่ก็ทำให้คนจริง/บอลตกหล่นได้ และยังแก้กรณีทำนายผิดคลาสด้วย conf สูงไม่ได้ ข้อผิดพลาดเสื้อเหลืองต้องมีข้อมูล Validation เพิ่มเติมที่ครอบคลุมกรณีนั้นด้วย

อ้างอิง predict/conf: https://docs.ultralytics.com/modes/predict/

In [ ]:
%pip install -q 'ultralytics==8.4.177' pillow pandas


In [ ]:
from pathlib import Path
from datetime import datetime
from importlib.metadata import version
import gc, hashlib, json, shutil, zipfile
import torch, pandas as pd
from ultralytics import YOLO
from google.colab import files
from PIL import Image as PILImage, ImageDraw
from IPython.display import display

assert torch.cuda.is_available(), 'ตั้ง Runtime → Change runtime type → T4 GPU ก่อน'
print('GPU:', torch.cuda.get_device_name(0))
RUN_ROOT = Path('/content/tactivision_confidence') / datetime.now().strftime('%Y%m%d_%H%M%S_%f')
RUN_ROOT.mkdir(parents=True, exist_ok=False)
CLASSES = ['ball', 'goalkeeper', 'player', 'referee']
THRESHOLDS = (0.15, 0.30, 0.50)
IMGSZ = 1280
NMS_IOU = 0.70
MATCH_IOU = 0.50


## อัปโหลด football-v2-test-ready.zip
ใช้ ZIP เดิมที่เตรียมให้ก่อนหน้านี้ ไฟล์เดียว ไม่มีการขอให้ Label ภาพใหม่

In [ ]:
uploaded = files.upload()
archives = [Path(name) for name in uploaded if name.lower().endswith('.zip')]
assert len(archives) == 1, 'เลือก football-v2-test-ready.zip เพียงไฟล์เดียว'
DATA_ROOT = RUN_ROOT / 'input'
DATA_ROOT.mkdir()
with zipfile.ZipFile(archives[0]) as archive:
    manifest = json.loads(archive.read('training_split_manifest.json'))
    audit = json.loads(archive.read('test_audit.json'))
    selected = [n for n in archive.namelist() if not n.endswith('/') and (
        n.startswith('valid/images/') or n.startswith('valid/labels/') or n == 'models/best.pt')]
    for name in selected:
        assert (DATA_ROOT / name).resolve().is_relative_to(DATA_ROOT.resolve()), 'พาธใน ZIP ไม่ถูกต้อง'
        archive.extract(name, DATA_ROOT)
del uploaded
valid_manifest = [row for row in manifest if row['split'] == 'valid']
validation_paths = sorted((DATA_ROOT / 'valid/images').glob('*.jpg'))
assert len(validation_paths) == len(valid_manifest) == 36
assert {p.name for p in validation_paths} == {row['filename'] for row in valid_manifest}
for row in valid_manifest:
    assert row['clip'] == 'tJnScrAKJfI', 'ใช้เฉพาะ Validation เดิม'
    image = DATA_ROOT / 'valid/images' / row['filename']
    assert hashlib.sha256(image.read_bytes()).hexdigest() == row['sha256']
    assert (DATA_ROOT / 'valid/labels' / (image.stem + '.txt')).is_file()
assert not (DATA_ROOT / 'test').exists() and not (DATA_ROOT / 'train').exists()
MODEL_PATH = DATA_ROOT / 'models/best.pt'
model_sha256 = hashlib.sha256(MODEL_PATH.read_bytes()).hexdigest()
assert model_sha256 == audit['best_model_sha256']
print('พร้อม: Validation',len(validation_paths),'ภาพ + best.pt เดิม ไม่มีภาพ Test ถูกแตกออกมา')


In [ ]:
def box_iou(a, b):
    intersection = max(0.0, min(a[2], b[2])-max(a[0], b[0])) * max(0.0, min(a[3], b[3])-max(a[1], b[1]))
    area_a = max(0.0,a[2]-a[0]) * max(0.0,a[3]-a[1])
    area_b = max(0.0,b[2]-b[0]) * max(0.0,b[3]-b[1])
    union = area_a + area_b - intersection
    return intersection / union if union > 0 else 0.0

def counts_at_threshold(predictions, truth, threshold, num_classes=4, match_iou=0.5):
    counts = [{'tp':0, 'fp':0, 'fn':0} for _ in range(num_classes)]
    used = set()
    accepted = sorted((p for p in predictions if p['confidence'] >= threshold),
                      key=lambda p:p['confidence'], reverse=True)
    for pred in accepted:
        class_id = pred['class_id']
        best_index, best_iou = None, -1.0
        for index, answer in enumerate(truth):
            if index in used or answer['class_id'] != class_id:
                continue
            overlap = box_iou(pred['xyxy'], answer['xyxy'])
            if overlap > best_iou:
                best_index, best_iou = index, overlap
        if best_index is not None and best_iou >= match_iou:
            counts[class_id]['tp'] += 1
            used.add(best_index)
        else:
            counts[class_id]['fp'] += 1
    for index, answer in enumerate(truth):
        if index not in used:
            counts[answer['class_id']]['fn'] += 1
    return counts

def precision_recall(counts):
    tp, fp, fn = counts['tp'], counts['fp'], counts['fn']
    return {'precision':tp/(tp+fp) if tp+fp else 0.0,
            'recall':tp/(tp+fn) if tp+fn else 0.0}


## รันโมเดลและสร้างภาพ 3 คอลัมน์
ตรวจภาพละหนึ่งครั้งที่ conf=0.15 แล้วกรองชุดผลเดิมเป็น 0.30 และ 0.50 โดยคง NMS และขนาดภาพเดิม บันทึกภาพเต็มสำหรับซูมครบ 36 ภาพ และแสดงตัวอย่าง 4 ภาพ

TP ต้องตรงทั้งตำแหน่งและคลาส: ตัวจริงเป็น player แต่ทำนายเป็น goalkeeper จะนับ FP ฝั่ง goalkeeper และ FN ฝั่ง player

In [ ]:
model = YOLO(str(MODEL_PATH))
assert [model.names[i] for i in range(len(model.names))] == CLASSES
OUTPUT = RUN_ROOT / 'results'
PANELS = OUTPUT / 'confidence_comparison'
PANELS.mkdir(parents=True)
totals = {threshold:[{'tp':0,'fp':0,'fn':0} for _ in CLASSES] for threshold in THRESHOLDS}
per_image, saved_predictions = [], []
for image_index, image_path in enumerate(validation_paths):
    result = model.predict(str(image_path), imgsz=IMGSZ, conf=min(THRESHOLDS), iou=NMS_IOU,
                           max_det=300, rect=True, augment=False, device=0, save=False, verbose=False)[0].cpu()
    height, width = result.orig_shape
    truth = []
    label_path = DATA_ROOT / 'valid/labels' / (image_path.stem + '.txt')
    for line in label_path.read_text().splitlines():
        if not line.strip():
            continue
        fields = line.split()
        assert len(fields) == 5, label_path.name
        class_id = int(fields[0])
        assert 0 <= class_id < len(CLASSES), label_path.name
        x,y,w,h = map(float,fields[1:])
        truth.append({'class_id':class_id,'xyxy':[(x-w/2)*width,(y-h/2)*height,(x+w/2)*width,(y+h/2)*height]})
    predictions = [{'xyxy':row[:4],'confidence':float(row[4]),'class_id':int(row[5])}
                   for row in result.boxes.data.tolist()]
    saved_predictions.append({'filename':image_path.name,'predictions':predictions})
    panel = PILImage.new('RGB',(width*len(THRESHOLDS),height+40),'white')
    draw = ImageDraw.Draw(panel)
    for column,threshold in enumerate(THRESHOLDS):
        counts = counts_at_threshold(predictions,truth,threshold,len(CLASSES),MATCH_IOU)
        for class_id,class_counts in enumerate(counts):
            for key in ('tp','fp','fn'):
                totals[threshold][class_id][key] += class_counts[key]
            per_image.append({'filename':image_path.name,'conf':threshold,'class':CLASSES[class_id],**class_counts})
        filtered = result[result.boxes.conf >= threshold]
        rendered = PILImage.fromarray(filtered.plot(line_width=1)[:, :, ::-1])
        panel.paste(rendered,(column*width,40))
        draw.text((column*width+10,12),f'conf={threshold:.2f} | detections={len(filtered.boxes)}',fill='black')
    panel.save(PANELS / image_path.name,quality=95)
    if image_index < 4:
        print(image_path.name.split('_jpg')[0])
        display(panel.resize((1536,round(panel.height*1536/panel.width))))
    print(f'[{image_index+1}/{len(validation_paths)}] {image_path.name.split("_jpg")[0]}')
del model
gc.collect()
torch.cuda.empty_cache()


In [ ]:
summary_rows, class_rows = [], []
for threshold in THRESHOLDS:
    total = {key:sum(row[key] for row in totals[threshold]) for key in ('tp','fp','fn')}
    summary_rows.append({'conf':threshold,**total,**precision_recall(total)})
    for class_id,counts in enumerate(totals[threshold]):
        class_rows.append({'conf':threshold,'class':CLASSES[class_id],**counts,**precision_recall(counts)})
summary = pd.DataFrame(summary_rows)
by_class = pd.DataFrame(class_rows)
print('รวมทุกคลาส: TP ถูก / FP เกิน / FN ตกหล่น / Precision และ Recall ที่ conf นั้นจริง ๆ')
display(summary.round(4))
print('แยกรายคลาส: ควรดู player และ ball ด้วย เพราะคะแนนรวมอาจบดบังปัญหาบางคลาส')
display(by_class.round(4))
summary.to_csv(OUTPUT / 'confidence_summary.csv',index=False)
by_class.to_csv(OUTPUT / 'confidence_by_class.csv',index=False)
pd.DataFrame(per_image).to_csv(OUTPUT / 'confidence_per_image.csv',index=False)
(OUTPUT / 'validation_predictions.json').write_text(json.dumps(saved_predictions,indent=2))
report = {'split':'validation','images':len(validation_paths),'thresholds':list(THRESHOLDS),
          'imgsz':IMGSZ,'nms_iou':NMS_IOU,'match_iou':MATCH_IOU,'model_sha256':model_sha256,
          'matching':'confidence-ordered greedy, class-aware, one-to-one; not mAP',
          'inference':'once at conf=0.15, filter same post-NMS detections for higher thresholds',
          'summary':summary_rows,'by_class':class_rows,
          'environment':{p:version(p) for p in ('ultralytics','torch','torchvision','numpy','pillow')}}
(OUTPUT / 'confidence_report.json').write_text(json.dumps(report,indent=2))
print('เลือกค่าจากทั้งตารางและภาพ โดยดูว่ากรอบมั่วลดลงคุ้มกับคนจริง/บอลที่หายไปไหม')


In [ ]:
result_zip = shutil.make_archive(str(RUN_ROOT / 'tactivision_confidence_results'),'zip',OUTPUT)
print('ส่งไฟล์นี้กลับมาให้ตรวจผลได้:',result_zip)
files.download(result_zip)
